# Load Dependencies

In [ ]:
# Data analysis libraries
import warnings; warnings.simplefilter(action='ignore', category=FutureWarning) # Ignoring FutureWarnings for the sake of this training exercise; not recommended for production code
import numpy as np
import pandas as pd; pd.options.display.max_columns = 200
import geopandas as gpd
import linref as lr
import pyproj

# Visualization libraries
import plotly.express as px

# Utility libraries
import os

In [ ]:
# Define global variables
PROJECT_CRS = pyproj.CRS.from_user_input('EPSG:3857')

# Load Data

**Download Sample Dataset**

You can download the sample dataset used in this training at <a href="https://tooledesign.egnyte.com/dl/7kCQctdWKQVV" target="_blank" rel="noopener noreferrer">this link</a>. Download the GeoPackage file and save it in the `99_Resources` folder. This is where the scripts expect it to be, so no adjustments need to be made to filepaths in the script files.

In [ ]:
# Point this to the location of the Geopackage file
fp = os.path.join('..', '99_Resources', 'franklin_county_training_data.gpkg')

# List all the layers in the file
gpd.list_layers(fp)

In [ ]:
# Load roadway data
roadways = gpd.read_file(fp, layer='roadways')
roadways.to_crs(PROJECT_CRS, inplace=True)

# Load crash data
query = """
SELECT OBJECTID, NLFID, COUNTY_LOG_NBR, CRASH_YR, KABCO, CRASH_TYPE_SIMPLE, DAY_IN_WEEK_TEXT, HOUR_PERIOD, geom
FROM crashes_enriched
"""
crashes = gpd.read_file(fp, sql=query)
crashes.to_crs(PROJECT_CRS, inplace=True)

print(f'Data loaded: {len(roadways):,.0f} roadways, {len(crashes):,.0f} crashes')

# Crash Scoring Metrics

In [ ]:
# Create a series of crash scoring metrics based on crash severity and mode
# These will be used for creating a variety of high-injury networks for each defined
# scoring metric

# We will first create boolean masks for each of the scoring metrics

# Crash severity metrics
mask_kabc = crashes['KABCO'].isin(['K', 'A', 'B', 'C'])
mask_ka   = crashes['KABCO'].isin(['K', 'A'])

# Crash mode metrics
mask_ped = crashes['CRASH_TYPE_SIMPLE'].isin(['Pedestrian'])
mask_pdc = crashes['CRASH_TYPE_SIMPLE'].isin(['Pedalcycle'])
mask_veh = ~(mask_ped | mask_pdc)

In [ ]:
# Combined metrics
crashes['KABC_VEH_SCORE'] = mask_kabc * mask_veh * 1
crashes['KABC_PED_SCORE'] = mask_kabc * mask_ped * 1
crashes['KABC_PDC_SCORE'] = mask_kabc * mask_pdc * 1
crashes['KA_VEH_SCORE']   = mask_ka   * mask_veh * 1
crashes['KA_PED_SCORE']   = mask_ka   * mask_ped * 1
crashes['KA_PDC_SCORE']   = mask_ka   * mask_pdc * 1

In [ ]:
# How can we create metrics for late night weekend crashes?
# E.g., Friday and Saturday nights between 9 PM and 3 AM

In [ ]:
crashes['KA_PED_SCORE'].sum()

In [ ]:
crashes.head()

# Linear Referencing
## What is Linear Referencing
Linear referencing is a system for storing, locating, and interpreting data based on their relative position along a static, principal linear feature without needing explicit geographic location information (such as latitude and longitude). These principal linear features, sometimes referred to as routes, are generally associated with actual spatial assets, such as roadways, railroads, or pipelines. 

Linear referencing allows users to locate additional assets with a geospatial component along the principal feature, such as point assets like signs, intersections, or crashes, as well as linear assets like guardrails, fences, or bridge structures. Additionally, linear referencing can be used to identify supplemental attributes about the principal linear feature which do not explicitly contain a geospatial component, such as the diameter of a pipe or the number of lanes along a roadway segment.

## Components of Linearly Referenced Data  

Linearly referenced data generally include four main components: 

1. **Route Identifier.** The route identifier indicates which linear geospatial feature will be referenced when computing the data point’s geographic location. This is a required component for all complete linearly referenced data sets. 

2. **Route Measure.** For point features, a single route measure is needed which indicates how far along the referenced route the feature is located. For linear features, both a begin and end measure are needed to indicate the bounds of the linear event. This is a required component for all complete linearly referenced data sets. 

3. **Event Attributes.** Additional descriptive information about the event and what it represents. This is not required for purely geospatial data but is a valuable component of many linearly referenced data sets. 

4. **Event Geometry.** Spatial geometry which represents the actual geographic location of the point or linear event. This is not required for non-geospatial events data tables which are linearly referenced to another data set which does include geometry. 

## Pulling Roadway Attributes to Crashes

In [ ]:
# Create events collections for managing linearly referenced data
# Roadways are linear events that have unique route IDs, a begin and end route distance measure, and geometry
roadways_ec = lr.EventsCollection(roadways, keys=['NLF_ID'], beg='CTL_BEGIN_', end='CTL_END_NB', geom='geometry')
# Crashes are point events that have unique route IDs, a locational route distance measure, and geometry
crashes_ec = lr.EventsCollection(crashes, keys=['NLFID'], beg='COUNTY_LOG_NBR', geom='geometry')

In [ ]:
# Using linear referencing, we can conflate information between two events collections on the 
# same linear referencing system
# The `linref` module can do this in a number of ways depending on the type of data
# All of these methods stem from the `.merge` method, which allows us to define a linear relationship
# between two events collections, producing an EventsMerge object which we an then query

# Define which columns we'd like to pull from the roadways data into the crashes data
get_columns = ['SPEED_LIMI', 'LANES_NBR', 'LANE_WIDTH']

# Create an EventsMerge object to define the linear relationship between the two events collections
em = crashes_ec.merge(roadways_ec)
em

In [ ]:
# Now that we have the EventsMerge object, we can query it to get the roadway data for each crash
# There are a variety of aggregators that we can use for this. Common ones include:
# - `mean` - takes the length-weighted mean of the overlaid events' data for each target event
# - `most` - takes the length-weighted mode of the overlaid events' data for each target event
# - `first` - takes the first intersecting event's attribute for each target event
# - `last` - takes the last intersecting event's attribute for each target event
# - `value_counts` - creates a dataframe of the counts of each overlaid event's attribute for each target event

# Let's start with the `first` aggregator since this makes the most sense for pulling roadway
# attributes into the crash data, assuming this will commonly be one-to-one
crashes_ec.df[get_columns] = em[get_columns].first()
crashes_ec.df.head(3)

In [ ]:
# Let's test this by looking at the attributes of the roadway that the first crash is on
# We can do this by indexing the route ID associated with the crash and accessing the 
# associated dataframe view
roadways_ec['CFRACR00025**C'].df.head(3)[['NLF_ID', 'CTL_BEGIN_', 'CTL_END_NB'] + get_columns]

## Pulling Crash Attributes to Roadways

In [ ]:
# Let's try flipping the events relationship around, pulling crash information to the roadways
# This time, we will use the `count` aggregator to get the number of crashes on each roadway

# Create the events merge object
em = roadways_ec.merge(crashes_ec)

# We can get the count of crashes on each roadway by using the `count` aggregator without needing
# to specify any columns
roadways_ec.df['KABCO_COUNT'] = em.count().fillna(0)
roadways_ec.df.sample(5, random_state=42)[['NLF_ID', 'CTL_BEGIN_', 'CTL_END_NB', 'KABCO_COUNT']]

In [ ]:
# Let's test this by looking at the crash data for the second roadway
# We can do this by indexing the route ID and begin/end mileposts associated with the segment
crashes_ec['SFRASR00665**C'].intersecting(beg=5.180, end=6.995, closed='left').shape

In [ ]:
# Instead of just getting the total number of crashes, we can also get the number of crashes
# broken out by another attribute like severity, for example

# Let's use the same events merge object as before, but this time we will use the `value_counts` aggregator
# Let's see what this produces on its own
em['KABCO'].value_counts().fillna(0)

In [ ]:
# We'll need to apply this to the roadways dataframe carefully to ensure that we accurately
# pull the columns we want in their correct order
# We will reorder the created columns to match our preferred order and then add them to 
# the roadways dataframe using new column names that follow the same order
right_columns = ['K', 'A', 'B', 'C']
left_columns = ['K_COUNT', 'A_COUNT', 'B_COUNT', 'C_COUNT']

# Pull the value_counts data into the roadways dataframe
roadways_ec.df[left_columns] = em['KABCO'].value_counts().fillna(0)[right_columns]

# Take a look at the results
roadways_ec.df.sample(5, random_state=42)[['NLF_ID', 'CTL_BEGIN_', 'CTL_END_NB', 'KABCO_COUNT'] + left_columns]

# Crash Distribution Analysis

Now, let's take this a bit further by applying a crash distribution aggregator that is somewhat similar to a sliding window analysis. This method distributes the impact of a crash among adjacent, equal-length segments, creating a smoothed crash distribution profile.

_Linref-based Crash Distribution Schematic Example_

<img src="../99_Resources/img/linref-distribution-schematic.png" width=1000>

## Resegmenting the Roadway

In [ ]:
# To perform this analysis, we need to resegment the roadways into short, equal-length segments
# First, we'll need to dissolve the roadways to create a single geometry for each which we can
# then segment into equal-length segments

# This is easily done using the `dissolve` method on our linearly referenced events collection
dissolved_ec = roadways_ec.dissolve()
dissolved_ec.df.explore()

In [ ]:
# That was easy! Now we can segment the roadway into equal-length segments using 
# the `to_windows` method. Let's use a segment length of 0.25 miles
segment_length = 0.25

# The `fill` parameter allows us to specify how to fill the last segment if it is shorter than the 
# specified length. There are a variety of options, most importantly:

# `cut` a truncated window will be created to fill the gap with a length less than the full window length.

# `extend` the final window will be anchored on the grid defined by the step value, extending beyond the
# window length to the right bound of the event.

# `balance` if the final range is greater than or equal to half the target range length, perform the cut 
# method; if it is less, perform the extend method.

# Let's perform the resegmentation
segments_ec = dissolved_ec.to_windows(length=segment_length, fill='balance')

# Let's take a look at a few segments that exemplify what we've done--note that the geometry is now gone
segments_ec.df.iloc[180:185]

In [ ]:
# We need to "re-cut" the geometry from the dissolved layer that we created based on the new segment bounds
# This goes back to the `merge` method we used earlier, now merging the dissolved layer with the segments
em = segments_ec.merge(dissolved_ec)

# We can use the `cut` method to cut the geometry from the dissolved layer into the segments
segments_ec.df['geometry'] = em.cut()

# Now, we just need to "promote" the dataframe into a geodataframe
segments_ec = segments_ec.cast_gdf(crs=PROJECT_CRS)

# Let's take a look at the segments we created
segments_ec.df.explore()

## Distributing Crashes onto the Resegmented Road

In [ ]:
# For later comparison, like before, let's quickly get a count of KABCO crashes on each new segment
em = segments_ec.merge(crashes_ec)
segments_ec.df['KABCO_COUNT'] = em.count().fillna(0)

In [ ]:
# Now that the segments are prepared, let's perform the crash distribution. Once again, we will
# use the `merge` method to create an events merge object that defines the linear relationship between
# the segments and the crashes
em = segments_ec.merge(crashes_ec)

# Now, we will use the specialized `distribute` method, which will perform a sliding window-type
# distribution of the crashes along the segments.
# There are a few parameters that we can adjust to configure the distribution:
segments_ec.df['KABCO_SCORE'] = em.distribute(
    # The number of segments to "blur" the crashes over; if this is set to 0, the crashes will be
    # distributed only to the segments they are on, producing a result similar to the `count` method
    blur_size=2,
    # The type of blur to apply; currently, the only option is 'linear', which applies a linear
    # distribution of the crashes over the segments, creating a pyramid-shaped distribution
    blur_style='linear',
).round(3)

# Let's take a look at the results
segments_ec.df.tail(6)[['NLF_ID', 'CTL_BEGIN_', 'CTL_END_NB', 'KABCO_COUNT', 'KABCO_SCORE']]

In [ ]:
# How does this distribution compare to the simple count of crashes on the segments?
print(segments_ec.df['KABCO_COUNT'].sum())
print(segments_ec.df['KABCO_SCORE'].sum())

In [ ]:
# Finally, let's use the advanced crash metrics that we created earlier to get more advanced
# crash distributions by mode and severity
# The only thing we need to change is to add a list of columns we'd like to analyze in the 
# `distribute` method

# Define the columns to distribute
distribute_columns = [
    'KABC_VEH_SCORE',
    'KABC_PED_SCORE',
    'KABC_PDC_SCORE',
    'KA_VEH_SCORE',
    'KA_PED_SCORE',
    'KA_PDC_SCORE',
]

# Perform the distribution using the same parameters as before
# Like with the `value_counts` method, we will need to be careful about selecting the columns
# and applying them to the dataframe in the correct order
distributed = em.distribute(
    column=distribute_columns,
    blur_size=2,
    blur_style='linear',
)
# Reorder the columns to match our preferred order
segments_ec.df[distribute_columns] = distributed[distribute_columns].round(3)

# Take a look at the results
segments_ec.df.tail(6)[['NLF_ID', 'CTL_BEGIN_', 'CTL_END_NB', 'KABCO_COUNT'] + distribute_columns]

## Binning Results into Tiers

In [ ]:
# Iterate over the scoring columns, creating new percentile and tier columns for each metric
for col in distribute_columns:
    # Create the new column using the `rank` method to get the percentile rank of each value
    percentile = segments_ec.df[col].rank(pct=True, ascending=True, method='min').round(3)

    # Now, we will assign tiers based on percentile values for each metric
    # The tiers are defined as follows:
    # - 0.0 - 0.50: Minimal
    # - 0.50 - 0.75: Low
    # - 0.75 - 0.85: Medium
    # - 0.85 - 0.95: High
    # - 0.90 - 0.95: Critical
    tiers = pd.cut(
        percentile,
        bins=[0, 0.5, 0.75, 0.85, 0.95, 1],
        labels=['Minimal', 'Low', 'Medium', 'High', 'Critical'],
        include_lowest=True,
    )

    # Add results as new columns in the segments dataframe
    segments_ec.df[f'{col}_PCT']  = percentile
    segments_ec.df[f'{col}_TIER'] = tiers

## Mapping the Results

In [ ]:
# Let's take a look at the results of this analysis on a map, visualizing based on the KABC_VEH_SCORE
VISUALIZE_COLUMN = 'KABC_VEH_SCORE'
segments_ec.df.explore(
    column=VISUALIZE_COLUMN,
    cmap='YlOrRd',
    vmax=segments_ec.df[VISUALIZE_COLUMN].quantile(0.95),
    popup=True,
    tooltip=False,
    style_kwds=dict(weight=3),
)

# Export Results

In [ ]:
# Export the high-injury network results to the same GPKG file
segments_ec.df.to_file(fp, layer='hin', driver='GPKG', index=False)